In [3]:
import altair as alt
import pandas as pd
from pathlib import Path

In [ ]:
DATA_DIR = Path().resolve().parent / "data"

combined_data_df = pd.read_csv(DATA_DIR / "cleaned/all_data_combined.csv")

combined_data_df["Years colonized"] = (
    combined_data_df.groupby("Entity")["Years colonized"].transform("max")
)

In [14]:
# Add country codes to the df for choropleths
import pycountry

def to_numeric(code):
    try:
        return int(pycountry.countries.get(alpha_3=code).numeric)
    except Exception:
        return None

combined_data_df["id"] = combined_data_df["Code"].map(to_numeric)

In [13]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

### Colonizer Color Mapping

In [52]:
colonizer_options = ["Belgium", "France", "Germany", "Italy", "Netherlands", "Portugal", "Spain", "United Kingdom", "Multiple colonizers", "Not colonized"]

colonizer_color_scheme = {
    "Belgium": "#ff68c0",
    "France": "#9100f2",
    "Germany": "#003df2",
    "Italy": "#00def2",
    "Netherlands": "#00c23a",
    "Portugal": "#f2e200",
    "Spain": "#ed8300",
    "United Kingdom": "#c70000",
    "Multiple colonizers": "#a1a1a1", 
    "Not colonized": "#3b3b3b"
}

### Legal Status of Homosexuality Choropleth

In [29]:
# color mapping

legal_status_colors = {
    "Legal": "#00cd3f",
    "Illegal": "#d10000",
    "null" : "#a1a1a1"
}

In [44]:
from vega_datasets import data

countries = alt.topo_feature(data.world_110m.url, "countries")

def legal_status_choropleth(df: pd.DataFrame, year: str):
    choropleth = (
        alt.Chart(countries)
        .mark_geoshape(stroke="white")
        .transform_lookup(
            lookup="id",
            from_=alt.LookupData(df, "id", ["Entity", "Legal status of homosexuality"]),
        )
        .encode(
            color=alt.Color(
                "Legal status of homosexuality:N",
                title="Legal Status",
                # scale=alt.Scale(
                #     domain=legal_status_colors.keys(),
                #     range=legal_status_colors.values()
                # )
                ),
        )
        .properties(title=year)
        .project(type="equalEarth")
    )

    return choropleth

In [ ]:
df_2000 = combined_data_df[combined_data_df["Year"] == 2000]
df_2025 = combined_data_df[combined_data_df["Year"] == 2025]

choropleth_2000 = legal_status_choropleth(df_2000, "2000")
choropleth_2025 = legal_status_choropleth(df_2025, "2025")

multiple_charts = choropleth_2000 | choropleth_2025
multiple_charts

### Number of years colonized choropleth

In [ ]:
# add a gray base to ensure all countries are displayed
countries = alt.topo_feature(data.world_110m.url, "countries")

def world_base():
    return (
        alt.Chart(countries)
        .mark_geoshape(fill="lightgray", stroke="white")
    )

years_colonized_choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke="white")
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_2025, "id", ["Entity", "Years colonized"]),
    )
    .encode(
        # fix this color scale
        color=alt.Color("Years colonized:Q", scale=alt.Scale(range=["#a3e8ef","#032281"])),
    )
    .properties(width=650, height=350, title="Number of years colonized")
)

chart = (world_base() + years_colonized_choropleth).project(type="equalEarth")
chart